In [3]:
#1. Scrap data  RUN CELL 1 AND CELL 4 AND ONWARDS
import requests
from bs4 import BeautifulSoup
import pandas as pd


urls = [
    "https://www.uia.no/english/studies/courses/2026/spring/ikt469.html",
    "https://www.uia.no/english/studies/courses/2026/spring/ikt460.html",
]

courses = []

for url in urls:

    # Download page
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    
    # Course title
    title = soup.find("h1").get_text(strip=True)

    # ECTS credits
    ects = soup.find(string="ECTS Credits:")
    ects_value = ects.find_next().get_text(strip=True)
    
    # Course leader
    leader = soup.find(string="Course Leader:")
    leader_value = leader.find_next().get_text(strip=True)
    
    """ # Teaching language
    language = soup.find(string="Teaching language")
    language_value = language.find_next("p").get_text(strip=True)"""
    
    # Learning outcomes section
    learning_section = soup.find("h2", string="Learning outcomes")
    learning_outcomes = learning_section.find_next("ul").get_text("\n", strip=True)
    
    # Contents section
    contents_section = soup.find("h2", string="Contents")
    contents_text = contents_section.find_next("p").get_text(strip=True)
    
    course_data = {
    "title": title,
    "ects": ects_value,
    "course_leader": leader_value,
    #"language": language_value,
    "learning_outcomes": learning_outcomes,
    "contents": contents_text
}
    courses.append(course_data)

# Save results
df = pd.DataFrame(courses)
df.to_csv("uia_ikt_courses.csv", index=False)

print(df)

                                         title ects   course_leader  \
0    IKT469 Deep Neural Networks (Spring 2026)  7.5  Morten Goodwin   
1  IKT460 Reinforcement Learning (Spring 2026)  7.5    Aditya Gupta   

                                   learning_outcomes  \
0  Understand advanced concepts in deep learning,...   
1  Understand the key features of reinforcement l...   

                                            contents  
0  The course covers theoretical and practical as...  
1  This course will teach the foundations of appl...  


In [4]:
#3. Embedding model and LLM
from sentence_transformers import SentenceTransformer
import ollama

# 1) Embedding model from Hugging Face
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# 2) LLM through Ollama
llm_model = "qwen2.5:0.5b"

# Test embedding
text = "IKT469 Deep Neural Networks covers transformers and multimodal learning."
embedding = embedding_model.encode(text)

print("Embedding length:", len(embedding))
print("First 5 values:", embedding[:5])

# Test LLM
response = ollama.chat(
    model=llm_model,
    messages=[
        {"role": "user", "content": "Summarize what a deep neural network is in 2 sentences."}
    ]
)

print(response["message"]["content"])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7235.80it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding length: 384
First 5 values: [-0.1556065  -0.05990236  0.07276607 -0.01477484  0.02260388]
A deep neural network (DNN) is a type of artificial intelligence model that mimics the structure and function of the human brain through interconnected nodes or "neurons." It consists of multiple layers, each containing one or more neurons, connected by weighted paths called connections. Each neuron analyzes input data to generate an output, often used for tasks such as image recognition, speech processing, and natural language understanding. The network's ability to learn from vast amounts of data is a key feature that sets DNNs apart from traditional models like convolutional neural networks (CNNs).


In [5]:
#4. Chunking and indexing RUN FROM HERE
import pandas as pd

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# 1) Load your scraped data
df = pd.read_csv("uia_ikt_courses.csv")

# 2) Turn each row into a LangChain Document
documents = []

for i, row in df.iterrows():
    title = str(row.get("title", ""))
    ects = str(row.get("ects", ""))
    leader = str(row.get("course_leader", ""))

    # If you also scraped more fields later, add them here
    content = f"""
Title: {row['title']}
ECTS: {row['ects']}
Course leader: {row['course_leader']}

Learning outcomes:
{row.get('learning_outcomes', '')}

Course contents:
{row.get('contents', '')}
""".strip()

    documents.append(
        Document(
            page_content=content,
            metadata={
                "row_id": i,
                "title": title,
                "ects": ects,
                "course_leader": leader,
            },
        )
    )

# 3) Chunk the documents
# For course descriptions, small chunks are usually enough
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(f"Original documents: {len(documents)}")
print(f"Chunks created: {len(chunks)}")

# 4) Create a CPU-only embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# 5) Index chunks in local Chroma
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_langchain_db",
    collection_name="uia_courses",
)

print("Chunking and indexing complete.")

/opt/miniconda3/envs/ikt469project/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


Original documents: 2
Chunks created: 16


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6360.53it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Chunking and indexing complete.


In [6]:
#5. RAG
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama

# 1) Load the same embedding model you used when indexing
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# 2) Load the persisted vector store
vectorstore = Chroma(
    persist_directory="./chroma_langchain_db",
    collection_name="uia_courses",
    embedding_function=embeddings,
)

# 3) Turn it into a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# 4) Load your Ollama LLM
llm = ChatOllama(
    model="qwen2.5:0.5b",   # or another model you actually pulled
    temperature=0
)

# 5) Ask a question
question = "What are the learning outcomes of the deep neural network course?"

# Retrieve relevant chunks
docs = retriever.invoke(question)

# Build context
context = "\n\n".join(doc.page_content for doc in docs)

# Prompt the LLM
prompt = f"""
Answer the question using only the context below.
If the answer is not in the context, say you do not know.

Context:
{context}

Question:
{question}
"""

response = llm.invoke(prompt)

print("QUESTION:")
print(question)
print("\nRETRIEVED DOCUMENTS:")
for i, doc in enumerate(docs, 1):
    print(f"\n--- Document {i} ---")
    print(doc.page_content)
    print("Metadata:", doc.metadata)

print("\nANSWER:")
print(response.content)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12635.29it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


QUESTION:
What are the learning outcomes of the deep neural network course?

RETRIEVED DOCUMENTS:

--- Document 1 ---
Learning outcomes:
Understand advanced concepts in deep learning, including why deep models generalize and how modern architectures are designed.
Metadata: {'title': 'IKT469 Deep Neural Networks (Spring 2026)', 'course_leader': 'Morten Goodwin', 'ects': '7.5', 'row_id': 0}

--- Document 2 ---
Title: IKT469 Deep Neural Networks (Spring 2026)
ECTS: 7.5
Course leader: Morten Goodwin
Metadata: {'course_leader': 'Morten Goodwin', 'ects': '7.5', 'row_id': 0, 'title': 'IKT469 Deep Neural Networks (Spring 2026)'}

ANSWER:
The learning outcomes of the deep neural network course are to understand advanced concepts in deep learning, including why deep models generalize and how modern architectures are designed.
